### Package import

In [18]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Prefer Kaggle input paths if available; fallback to local data/raw.
kaggle_dir = Path("/kaggle/input/competitions/playground-series-s6e2")
if kaggle_dir.exists():
    train = pd.read_csv(kaggle_dir / "train.csv")
    test = pd.read_csv(kaggle_dir / "test.csv")
    original = pd.read_csv("/kaggle/input/datasets/neurocipher/heartdisease/Heart_Disease_Prediction.csv") 
    sub = pd.read_csv(kaggle_dir / "sample_submission.csv")
else:
    # Infer project root (so it works even when run from nb/)
    cwd = Path.cwd().resolve()
    project_root = None
    for p in [cwd] + list(cwd.parents):
        if (p / "data" / "raw" / "train.csv").exists():
            project_root = p
            break
    if project_root is None:
        raise FileNotFoundError("project root not found: data/raw/train.csv")

    data_dir = project_root / "data" / "raw"
    train = pd.read_csv(data_dir / "train.csv")
    test = pd.read_csv(data_dir / "test.csv")
    sub = pd.read_csv(data_dir / "sample_submission.csv")

print(train.shape, test.shape, sub.shape)


(630000, 15) (270000, 14) (270000, 2)


In [19]:
!pip install pytabkit -q

In [20]:
import torch
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from pytabkit import RealMLP_TD_Classifier
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Libraries loaded successfully.


### Data download

In [21]:
display(train.head())
display(test.head())
display(sub.head())

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7
3,630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3
4,630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7


,id,Heart Disease
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0


In [22]:
# Shapes
print("train:", train.shape)
print("test:", test.shape)

# Column diffs
train_cols = set(train.columns)
test_cols = set(test.columns)
print("Only in train:", sorted(train_cols - test_cols))
print("Only in test:", sorted(test_cols - train_cols))

# dtypes
train.dtypes.to_frame("dtype").head(30)


train: (630000, 15)
test: (270000, 14)
Only in train: ['Heart Disease']
Only in test: []


,dtype
id,int64
Age,int64
Sex,int64
Chest pain type,int64
BP,int64
Cholesterol,int64
FBS over 120,int64
EKG results,int64
Max HR,int64
Exercise angina,int64


This notebook uses the **RealMLP (Regularized MLP)** model from the `pytabkit` library to predict heart disease. The key features include:

- **External Data Feature Engineering**: Target statistics (mean, median, std, skew, count) from the original UCI dataset
- **5-Fold Stratified Cross-Validation**
- **Optimized Hyperparameters** for best performance

| Column Name | Description | Type | Typical Range / Values | Clinical Relevance |
|-------------|-------------|------|------------------------|--------------------|
| 🧓 **Age** | Age of the patient in years | Numeric | 29–77 | Risk increases after age 45 (men) / 55 (women) |
| 🚹 **Sex** | Gender of the patient | Binary | 1=Male, 0=Female | Men have higher risk at younger ages |
| 💔 **Chest pain type** | Type of chest pain experienced | Categorical | 1–4 | Typical angina (1) and Asymptomatic (4) indicate higher risk |
| 💉 **BP** | Resting blood pressure | Numeric | 94–200 mm Hg | Hypertension (>140/90) is a major risk factor |
| 🧈 **Cholesterol** | Serum cholesterol level | Numeric | 126–564 mg/dL | High levels (>200-240) promote plaque buildup |
| 🍬 **FBS over 120** | Fasting blood sugar > 120 mg/dL | Binary | 1=True, 0=False | Diabetes is a strong CAD risk factor |
| 📈 **EKG results** | Resting ECG results | Categorical | 0–2 | Indicates possible ischemia or structural changes |
| ❤️ **Max HR** | Maximum heart rate achieved | Numeric | 71–202 bpm | Lower max HR indicates poor cardiac reserve |
| 🏃 **Exercise angina** | Exercise-induced angina | Binary | 1=Yes, 0=No | Very specific sign of ischemia |
| 📉 **ST depression** | ST depression during exercise | Numeric | 0.0–6.2 | Values ≥1.0 mm are often diagnostic |
| ⛰️ **Slope of ST** | Slope of peak exercise ST segment | Categorical | 1–3 | Downsloping (3) indicates severe ischemia |
| 🩸 **Number of vessels fluro** | Major coronary vessels colored by fluoroscopy | Ordinal | 0–3 | Higher values = more severe CAD |
| 🧬 **Thallium** | Thallium stress test result | Categorical | 3,6,7 | Reversible defects indicate ischemia |
| 🎯 **Heart Disease** | Target variable | Binary | Presence/Absence | ≥50% narrowing in at least one major artery |

In [23]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_STATE = 42
N_FOLDS = 5

print(f"Using device: {DEVICE}")

Using device: cpu


# <div style="text-align:center; border-radius:15px; padding:15px; background: linear-gradient(90deg, #0d9488, #06b6d4); color:white; font-family: Arial Black;"><b>3. Data Loading and Preprocessing</b></div>

In [24]:
def encode_target_strict(y: pd.Series) -> pd.Series:
    mapping_candidates = [
        {"No": 0, "Yes": 1},
        {"N": 0, "Y": 1},
        {"Negative": 0, "Positive": 1},
        {"Absent": 0, "Present": 1},
        {"Absence": 0, "Presence": 1},
        {0: 0, 1: 1},
        {"0": 0, "1": 1},
    ]

    uniq = set(y.dropna().unique().tolist())

    for mp in mapping_candidates:
        if uniq.issubset(set(mp.keys())):
            return y.map(mp).astype("int8")

    raise ValueError(f"Unknown target labels: {sorted(list(uniq))}")


train["Heart Disease"] = encode_target_strict(train["Heart Disease"])
if "Heart Disease" in original.columns:
    original["Heart Disease"] = encode_target_strict(original["Heart Disease"])

base_features = [col for col in train.columns if col not in ["Heart Disease", "id"]] 

def add_engineered_features(df):
    df_temp = df.copy()
    initial_rows = len(df_temp)
    
    for col in base_features: 
        if col in original.columns:
           
            stats = original.groupby(col)["Heart Disease"].agg(["mean", "median", "std", "skew", "count"]).reset_index()
         
            stats.columns = [col] + [f"orig_{col}_{s}" for s in ["mean", "median", "std", "skew", "count"]]
     
            df_temp = df_temp.merge(stats, on=col, how="left") 
            if len(df_temp) != initial_rows:
                raise ValueError(f"Merge expanded rows for column {col}! Initial: {initial_rows}, Current: {len(df_temp)}")
 
            fill_values = {
                f"orig_{col}_mean": original["Heart Disease"].mean(),
                f"orig_{col}_median": original["Heart Disease"].median(),
                f"orig_{col}_std": 0,
                f"orig_{col}_skew": 0,
                f"orig_{col}_count": 0
            }
            df_temp = df_temp.fillna(value=fill_values)
            
    return df_temp


def normalize_dtypes(train_df: pd.DataFrame, test_df: pd.DataFrame, target_col: str, id_col: str | None = "id"):
    drop_cols = [c for c in [target_col, id_col] if c in train_df.columns]

    X_tr = train_df.drop(columns=drop_cols)
    X_te = test_df.drop(columns=[c for c in [id_col] if c in test_df.columns])

    obj_cols = X_tr.select_dtypes(include=["object", "string"]).columns.tolist()
    for c in obj_cols:
        tr_num = pd.to_numeric(X_tr[c], errors="coerce")
        te_num = pd.to_numeric(X_te[c], errors="coerce")
        tr_ok = tr_num.notna().mean()
        te_ok = te_num.notna().mean()
        if min(tr_ok, te_ok) >= 0.98:
            X_tr[c] = tr_num.astype("float32")
            X_te[c] = te_num.astype("float32")

    cat_cols = X_tr.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    for c in cat_cols:
        X_tr[c] = X_tr[c].astype("category")
        X_te[c] = X_te[c].astype("category")

    num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    for c in num_cols:
        X_tr[c] = X_tr[c].astype("float32")
        X_te[c] = X_te[c].astype("float32")

    return X_tr, X_te, cat_cols


train = add_engineered_features(train)
test = add_engineered_features(test) 

print(f"Train Shape after FE: {train.shape}")
print(f"Test Shape after FE: {test.shape}")

if len(train) != 630000:
    print("WARNING: Train row count mismatch!")
if len(test) != 270000:
    print("WARNING: Test row count mismatch!")

X, X_test, cat_cols = normalize_dtypes(train, test, target_col="Heart Disease", id_col="id")
y = train["Heart Disease"]

print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")
print(f"X Shape: {X.shape}")
print(f"X_test Shape: {X_test.shape}")
print(f"Number of categorical cols: {len(cat_cols)}")


ValueError: Unknown target labels: ['Absence', 'Presence']

# <div style="text-align:center; border-radius:15px; padding:15px; background: linear-gradient(90deg, #0d9488, #06b6d4); color:white; font-family: Arial Black;"><b>4. Data Quality Check</b></div>

In [ ]:
def check_data_quality(df, name="Dataset"):
    print(f"--- Data Quality: {name} ---")
    print(f"Total Rows: {len(df)}")

    cols_to_check = [c for c in df.columns if c != 'id']
    dupes = df.duplicated(subset=cols_to_check).sum()

    nan_counts = df.isnull().sum()
    total_nans = nan_counts.sum()
    
    print(f"Duplicate Rows (excl. ID): {dupes}")
    print(f"Total NaN values: {total_nans}")
    if total_nans > 0:
        print("\nColumns with NaNs:")
        print(nan_counts[nan_counts > 0])
    print("-" * 30)

check_data_quality(train, "Train")
check_data_quality(test, "Test")

# <div style="text-align:center; border-radius:15px; padding:15px; background: linear-gradient(90deg, #0d9488, #06b6d4); color:white; font-family: Arial Black;"><b>5. Feature Uniqueness & Cardinality</b></div>

In [ ]:
def analyze_uniqueness(df):
    unique_stats = []
    for col in df.columns:
        if col == 'id': continue
        
        n_unique = df[col].nunique()
        dtype = df[col].dtype

        category_guess = "Categorical/Ordinal" if n_unique < 25 else "Continuous"
        
        unique_stats.append({
            'Feature': col,
            'Unique Values': n_unique,
            'Data Type': dtype,
            'Heuristic Type': category_guess
        })
    
    return pd.DataFrame(unique_stats).sort_values(by='Unique Values')

uniqueness_df = analyze_uniqueness(train)
uniqueness_df

# <div style="text-align:center; border-radius:15px; padding:15px; background: linear-gradient(90deg, #0d9488, #06b6d4); color:white; font-family: Arial Black;"><b>6. Visualize Top Skewed Features</b></div>

In [ ]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

skew_series = X[numeric_cols].skew().abs().sort_values(ascending=False)
top_skewed_features = skew_series.head(6).index.tolist()

print("Top 6 Most Skewed Features (Absolute Values):")
print(X[top_skewed_features].skew())

plt.figure(figsize=(16, 10))
for i, col in enumerate(top_skewed_features):
    plt.subplot(2, 3, i + 1) 
    sns.histplot(X[col].sample(min(10000, len(X))), kde=True, color='teal', bins=30)
    plt.title(f"Feature: {col}\nSkewness: {X[col].skew():.2f}")
    plt.xlabel("")
    plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

# <div style="text-align:center; border-radius:15px; padding:15px; background: linear-gradient(90deg, #0d9488, #06b6d4); color:white; font-family: Arial Black;"><b>7. Cross-Validation & Training</b></div>

In [ ]:
%%time

param_grid = {
        'device': 'cuda',
        'random_state': 42,
        'verbosity': 2,
        'n_epochs': 100,
        'batch_size': 256, 
        'n_ens': 8, 
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 8,
        'first_layer_lr_factor': 0.5962121993798933,
        'val_metric_name': '1-auc_ovr',
        'hidden_sizes': "rectangular",
        'hidden_width': 384,
        'lr': 0.04, 
        'ls_eps': 0.011498317194338772,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 4, 
        'p_drop': 0.07301419697186451,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16, 
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    } 

def make_model(param_grid, cat_cols):
    try:
        return RealMLP_TD_Classifier(**param_grid, cat_col_names=cat_cols)
    except TypeError:
        # Older pytabkit versions may not support cat_col_names
        return RealMLP_TD_Classifier(**param_grid)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_scores = [] 

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Starting Fold {fold + 1} ---")

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx] 

    model = make_model(param_grid, cat_cols) 
    model.fit(X_tr, y_tr.values, X_val, y_val.values) 

    val_probs = model.predict_proba(X_val)[:, 1] 
    fold_test_probs = model.predict_proba(X_test)[:, 1] 

    oof_preds[val_idx] = val_probs
    test_preds += fold_test_probs / N_FOLDS

    score = roc_auc_score(y_val, val_probs)
    fold_scores.append(score)
    print(f"Fold {fold + 1} ROC-AUC Score: {score:.5f}")

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# <div style="text-align:center; border-radius:15px; padding:15px; background: linear-gradient(90deg, #0d9488, #06b6d4); color:white; font-family: Arial Black;"><b>8. Model Evaluation</b></div>

In [ ]:
from sklearn.calibration import CalibrationDisplay

plt.figure(figsize=(12, 5)) 
plt.suptitle('RealMLP', fontsize=20) 

ax1 = plt.subplot(1, 2, 1)
CalibrationDisplay.from_predictions(y, oof_preds, n_bins=100, strategy='quantile', ax=ax1)
ax1.set_title('Calibration Curve')

ax2 = plt.subplot(1, 2, 2)
ax2.hist(oof_preds, bins=100, edgecolor='black', alpha=0.7)
ax2.set_title('Prediction Distribution (Histogram)')
ax2.set_xlabel('Predicted Probability')
ax2.set_ylabel('Count')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
plt.show()

# <div style="text-align:center; border-radius:15px; padding:15px; background: linear-gradient(90deg, #0d9488, #06b6d4); color:white; font-family: Arial Black;"><b>9. Evaluation and Submission</b></div>

In [ ]:
total_oof_score = roc_auc_score(y, oof_preds) 

print("\n" + "="*40)
print(f"Overall OOF ROC-AUC: {total_oof_score:.5f}")
print(f"Mean Fold Score: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")
print("="*40)

In [ ]:
# Fold Scores Table
fold_df = pd.DataFrame({
    'Fold': [f'Fold {i+1}' for i in range(N_FOLDS)],
    'ROC-AUC': fold_scores
})
fold_df

In [ ]:
# pd.DataFrame({'id': train['id'], 'Heart Disease_prob': oof_preds}).to_csv('oof_preds_train.csv', index=False)
# commented out oof generation to strict prevention of submitting the wrong file

submission = pd.DataFrame({'id': test['id'], 'Heart Disease': test_preds})
submission.to_csv('submission.csv', index=False)

print('Submission saved!')
print(f'Shape: {submission.shape}')

if len(submission) != 270000:
    raise ValueError(f"CRITICAL ERROR: Submission row count {len(submission)} != expected 270000")

submission.head()

<div style="background: linear-gradient(90deg, #0d9488, #06b6d4); padding: 20px; border-radius: 10px; text-align: center; color: white; font-size: 18px; font-weight: bold;">
Thank you for reading! If you find this notebook useful, please consider giving it an upvote. 😊
</div>